In [1]:
import pandas as pd
import os

In [2]:
data_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Raw_Data"

In [3]:
os.listdir(data_path)

['accounts.csv',
 'account_statuses.csv',
 'account_types.csv',
 'addresses.csv',
 'branches.csv',
 'customers.csv',
 'customer_types.csv',
 'loans.csv',
 'loan_statuses.csv',
 'readme.md',
 'transactions.csv',
 'transaction_types.csv']

In [4]:
csv_files = [
    file for file in os.listdir(data_path)
    if file.lower().endswith(".csv")
]

print("Number of CSV files:", len(csv_files))
print(csv_files)

Number of CSV files: 11
['accounts.csv', 'account_statuses.csv', 'account_types.csv', 'addresses.csv', 'branches.csv', 'customers.csv', 'customer_types.csv', 'loans.csv', 'loan_statuses.csv', 'transactions.csv', 'transaction_types.csv']


In [5]:
data = {}

for file in csv_files:
    file_path = os.path.join(data_path, file)
    table_name = os.path.splitext(file)[0]
    data[table_name] = pd.read_csv(file_path)

print(data.keys())

dict_keys(['accounts', 'account_statuses', 'account_types', 'addresses', 'branches', 'customers', 'customer_types', 'loans', 'loan_statuses', 'transactions', 'transaction_types'])


In [6]:
# Basic Profiling of All Datasets

for name, df in data.items():
    
    print("=" * 60)
    print("DATASET:", name)
    print("=" * 60)
    
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Size:", df.size)
    
    print("\nColumn Names:")
    print(df.columns.tolist())
    
    print("\nData Types:")
    print(df.dtypes)
    
    print("\nMissing Values:")
    print(df.isnull().sum())
    
    print("\nDuplicate Rows:", df.duplicated().sum())
    
    print("\n" + "=" * 60)
    print()

DATASET: accounts
Rows: 1667
Columns: 6
Size: 10002

Column Names:
['AccountID', 'CustomerID', 'AccountTypeID', 'AccountStatusID', 'Balance', 'OpeningDate']

Data Types:
AccountID            int64
CustomerID           int64
AccountTypeID        int64
AccountStatusID      int64
Balance            float64
OpeningDate         object
dtype: object

Missing Values:
AccountID           0
CustomerID          0
AccountTypeID       0
AccountStatusID     0
Balance             0
OpeningDate        33
dtype: int64

Duplicate Rows: 16


DATASET: account_statuses
Rows: 3
Columns: 2
Size: 6

Column Names:
['AccountStatusID', 'StatusName']

Data Types:
AccountStatusID     int64
StatusName         object
dtype: object

Missing Values:
AccountStatusID    0
StatusName         0
dtype: int64

Duplicate Rows: 0


DATASET: account_types
Rows: 5
Columns: 2
Size: 10

Column Names:
['AccountTypeID', 'TypeName']

Data Types:
AccountTypeID     int64
TypeName         object
dtype: object

Missing Values:
AccountT

In [7]:
# Overall Data Quality Summary

dq_summary = []

for name, df in data.items():
    
    dq_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": df.isnull().sum().sum(),
        "Duplicate_Rows": df.duplicated().sum()
    })

dq_summary = pd.DataFrame(dq_summary)

dq_summary

,Dataset,Rows,Columns,Missing_Values,Duplicate_Rows
0,accounts,1667,6,33,16
1,account_statuses,3,2,0,0
2,account_types,5,2,0,0
3,addresses,1222,4,74,12
4,branches,50,3,0,0
5,customers,1111,6,45,11
6,customer_types,3,2,0,0
7,loans,333,7,12,3
8,loan_statuses,3,2,0,0
9,transactions,50000,8,1000,500


In [8]:
# Column-Level Data Quality Analysis

for name, df in data.items():
    
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    
    if len(missing) > 0:
        print("=" * 60)
        print("DATASET:", name)
        print("=" * 60)
        
        print("\nMissing Values by Column:")
        print(missing)
        
        print()

DATASET: accounts

Missing Values by Column:
OpeningDate    33
dtype: int64

DATASET: addresses

Missing Values by Column:
Street     24
City       26
Country    24
dtype: int64

DATASET: customers

Missing Values by Column:
FirstName    22
LastName     23
dtype: int64

DATASET: loans

Missing Values by Column:
StartDate           6
EstimatedEndDate    6
dtype: int64

DATASET: transactions

Missing Values by Column:
TransactionDate    1000
dtype: int64



In [9]:
# Duplicate Identifier Analysis

key_columns = {
    "accounts": "AccountID",
    "addresses": "AddressID",
    "branches": "BranchID",
    "customers": "CustomerID",
    "loans": "LoanID",
    "transactions": "TransactionID"
}

for table, column in key_columns.items():
    
    df = data[table]
    
    duplicate_count = df[column].duplicated().sum()
    null_count = df[column].isnull().sum()
    
    print("=" * 60)
    print("DATASET:", table)
    print("KEY COLUMN:", column)
    print("Duplicate IDs:", duplicate_count)
    print("Missing IDs:", null_count)

DATASET: accounts
KEY COLUMN: AccountID
Duplicate IDs: 16
Missing IDs: 0
DATASET: addresses
KEY COLUMN: AddressID
Duplicate IDs: 12
Missing IDs: 0
DATASET: branches
KEY COLUMN: BranchID
Duplicate IDs: 0
Missing IDs: 0
DATASET: customers
KEY COLUMN: CustomerID
Duplicate IDs: 11
Missing IDs: 0
DATASET: loans
KEY COLUMN: LoanID
Duplicate IDs: 3
Missing IDs: 0
DATASET: transactions
KEY COLUMN: TransactionID
Duplicate IDs: 500
Missing IDs: 0


In [10]:
# Inspect Duplicate Transaction IDs

transactions = data["transactions"]

duplicate_transactions = transactions[
    transactions["TransactionID"].duplicated(keep=False)
].sort_values("TransactionID")

duplicate_transactions.head(20)

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
33986,3000044,201292,200088,2,1644.10,2022-11-21 11:00:00.000000,26,Transaction 44
27816,3000044,201292,200088,2,1644.10,2022-11-21 11:00:00.000000,26,Transaction 44
42868,3000051,200946,201528,1,1127.32,2024-01-27 18:00:00.000000,32,Transaction 51
27578,3000051,200946,201528,1,1127.32,2024-01-27 18:00:00.000000,32,Transaction 51
34758,3000098,200673,201509,2,175.29,2021-01-02 00:00:00.000000,48,Transaction 98
44486,3000098,200673,201509,2,175.29,2021-01-02 00:00:00.000000,48,Transaction 98
6384,3000145,200232,201371,4,4700.18,2025-10-04 15:01:43.084879,25,Transaction 145
34004,3000145,200232,201371,4,4700.18,2025-10-04 15:01:43.084879,25,Transaction 145
19838,3000164,200756,201620,1,2281.73,2023-08-04 07:00:00.000000,42,Transaction 164
43793,3000164,200756,201620,1,2281.73,2023-08-04 07:00:00.000000,42,Transaction 164


In [11]:
# Exact Duplicate Analysis for Transactions

transactions = data["transactions"]

exact_duplicate_count = transactions.duplicated().sum()

print("Total Transaction Rows:", len(transactions))
print("Exact Duplicate Rows:", exact_duplicate_count)
print("Unique Transaction IDs:", transactions["TransactionID"].nunique())
print("Duplicate Transaction IDs:", transactions["TransactionID"].duplicated().sum())

Total Transaction Rows: 50000
Exact Duplicate Rows: 500
Unique Transaction IDs: 49500
Duplicate Transaction IDs: 500


In [12]:
# Referential Integrity Checks

transactions = data["transactions"]
accounts = data["accounts"]
customers = data["customers"]
branches = data["branches"]
transaction_types = data["transaction_types"]

checks = {
    "Transaction → Origin Account": (
        transactions["AccountOriginID"],
        accounts["AccountID"]
    ),
    
    "Transaction → Destination Account": (
        transactions["AccountDestinationID"],
        accounts["AccountID"]
    ),
    
    "Transaction → Transaction Type": (
        transactions["TransactionTypeID"],
        transaction_types["TransactionTypeID"]
    ),
    
    "Transaction → Branch": (
        transactions["BranchID"],
        branches["BranchID"]
    ),
    
    "Account → Customer": (
        accounts["CustomerID"],
        customers["CustomerID"]
    )
}

for check_name, (child_column, parent_column) in checks.items():
    
    invalid_count = (~child_column.isin(parent_column)).sum()
    
    print("=" * 60)
    print(check_name)
    print("Invalid References:", invalid_count)

Transaction → Origin Account
Invalid References: 0
Transaction → Destination Account
Invalid References: 0
Transaction → Transaction Type
Invalid References: 0
Transaction → Branch
Invalid References: 0
Account → Customer
Invalid References: 0


In [13]:
# Transaction Amount Validation

transactions = data["transactions"]

negative_amounts = (transactions["Amount"] < 0).sum()
zero_amounts = (transactions["Amount"] == 0).sum()

print("Negative Amounts:", negative_amounts)
print("Zero Amounts:", zero_amounts)

Negative Amounts: 0
Zero Amounts: 0


In [14]:
# Transaction Date Validation

transactions = data["transactions"].copy()

transactions["TransactionDate"] = pd.to_datetime(
    transactions["TransactionDate"],
    errors="coerce"
)

invalid_dates = transactions["TransactionDate"].isna().sum()

print("Invalid / Unparseable Transaction Dates:", invalid_dates)
print("Earliest Transaction Date:", transactions["TransactionDate"].min())
print("Latest Transaction Date:", transactions["TransactionDate"].max())

Invalid / Unparseable Transaction Dates: 1000
Earliest Transaction Date: 2020-01-01 00:00:00
Latest Transaction Date: 2026-08-28 15:01:43.084879


In [15]:
# Future Transaction Date Validation

today = pd.Timestamp.today()

future_dates = transactions[
    transactions["TransactionDate"] > today
]

print("Today's Date:", today)
print("Future Transaction Dates:", len(future_dates))

Today's Date: 2026-08-21 20:28:16.867488
Future Transaction Dates: 17


In [16]:
# Loan Date Validation

loans = data["loans"].copy()

loans["StartDate"] = pd.to_datetime(
    loans["StartDate"],
    errors="coerce"
)

loans["EstimatedEndDate"] = pd.to_datetime(
    loans["EstimatedEndDate"],
    errors="coerce"
)

invalid_date_order = loans[
    (loans["StartDate"].notna()) &
    (loans["EstimatedEndDate"].notna()) &
    (loans["EstimatedEndDate"] < loans["StartDate"])
]

print("Invalid Loan Date Order:", len(invalid_date_order))

Invalid Loan Date Order: 1


In [17]:
# Inspect Invalid Loan Date Record

invalid_date_order[
    [
        "LoanID",
        "AccountID",
        "StartDate",
        "EstimatedEndDate"
    ]
]

,LoanID,AccountID,StartDate,EstimatedEndDate
262,400080,201058,2026-03-18 15:01:44.338382,2024-05-30


In [18]:
# Actual Data Quality Findings

dq_findings = pd.DataFrame([
    {
        "Dataset": "accounts",
        "Column": "OpeningDate",
        "DQ_Dimension": "Completeness",
        "Issue": "Missing Values",
        "Failed_Records": 33,
        "Severity": "Medium",
        "Treatment": "Flag for review"
    },
    {
        "Dataset": "addresses",
        "Column": "Street, City, Country",
        "DQ_Dimension": "Completeness",
        "Issue": "Missing Values",
        "Failed_Records": 74,
        "Severity": "Medium",
        "Treatment": "Flag for review"
    },
    {
        "Dataset": "customers",
        "Column": "FirstName, LastName",
        "DQ_Dimension": "Completeness",
        "Issue": "Missing Values",
        "Failed_Records": 45,
        "Severity": "Medium",
        "Treatment": "Flag for review"
    },
    {
        "Dataset": "loans",
        "Column": "StartDate, EstimatedEndDate",
        "DQ_Dimension": "Completeness",
        "Issue": "Missing Values",
        "Failed_Records": 12,
        "Severity": "Medium",
        "Treatment": "Flag for review"
    },
    {
        "Dataset": "transactions",
        "Column": "TransactionDate",
        "DQ_Dimension": "Completeness",
        "Issue": "Missing Values",
        "Failed_Records": 1000,
        "Severity": "High",
        "Treatment": "Flag for review"
    },
    {
        "Dataset": "accounts",
        "Column": "AccountID",
        "DQ_Dimension": "Uniqueness",
        "Issue": "Duplicate IDs",
        "Failed_Records": 16,
        "Severity": "High",
        "Treatment": "Investigate duplicates"
    },
    {
        "Dataset": "addresses",
        "Column": "AddressID",
        "DQ_Dimension": "Uniqueness",
        "Issue": "Duplicate IDs",
        "Failed_Records": 12,
        "Severity": "High",
        "Treatment": "Investigate duplicates"
    },
    {
        "Dataset": "customers",
        "Column": "CustomerID",
        "DQ_Dimension": "Uniqueness",
        "Issue": "Duplicate IDs",
        "Failed_Records": 11,
        "Severity": "High",
        "Treatment": "Investigate duplicates"
    },
    {
        "Dataset": "loans",
        "Column": "LoanID",
        "DQ_Dimension": "Uniqueness",
        "Issue": "Duplicate IDs",
        "Failed_Records": 3,
        "Severity": "High",
        "Treatment": "Investigate duplicates"
    },
    {
        "Dataset": "transactions",
        "Column": "TransactionID",
        "DQ_Dimension": "Uniqueness",
        "Issue": "Exact Duplicate Rows",
        "Failed_Records": 500,
        "Severity": "Critical",
        "Treatment": "Remove exact duplicates during ETL"
    },
    {
        "Dataset": "transactions",
        "Column": "TransactionDate",
        "DQ_Dimension": "Validity",
        "Issue": "Future Dates",
        "Failed_Records": 17,
        "Severity": "Medium",
        "Treatment": "Flag for business review"
    },
    {
        "Dataset": "loans",
        "Column": "StartDate, EstimatedEndDate",
        "DQ_Dimension": "Consistency",
        "Issue": "End Date Before Start Date",
        "Failed_Records": 1,
        "Severity": "High",
        "Treatment": "Flag for business review"
    }
])

dq_findings

,Dataset,Column,DQ_Dimension,Issue,Failed_Records,Severity,Treatment
0,accounts,OpeningDate,Completeness,Missing Values,33,Medium,Flag for review
1,addresses,"Street, City, Country",Completeness,Missing Values,74,Medium,Flag for review
2,customers,"FirstName, LastName",Completeness,Missing Values,45,Medium,Flag for review
3,loans,"StartDate, EstimatedEndDate",Completeness,Missing Values,12,Medium,Flag for review
4,transactions,TransactionDate,Completeness,Missing Values,1000,High,Flag for review
5,accounts,AccountID,Uniqueness,Duplicate IDs,16,High,Investigate duplicates
6,addresses,AddressID,Uniqueness,Duplicate IDs,12,High,Investigate duplicates
7,customers,CustomerID,Uniqueness,Duplicate IDs,11,High,Investigate duplicates
8,loans,LoanID,Uniqueness,Duplicate IDs,3,High,Investigate duplicates
9,transactions,TransactionID,Uniqueness,Exact Duplicate Rows,500,Critical,Remove exact duplicates during ETL


In [19]:
# ETL Cleaning - Remove Exact Duplicate Transactions

transactions_clean = data["transactions"].copy()

before_count = len(transactions_clean)

transactions_clean = transactions_clean.drop_duplicates()

after_count = len(transactions_clean)

print("Rows Before Cleaning:", before_count)
print("Rows Removed:", before_count - after_count)
print("Rows After Cleaning:", after_count)

Rows Before Cleaning: 50000
Rows Removed: 500
Rows After Cleaning: 49500


In [20]:
# ETL Transformation - Flag Missing Transaction Dates

transactions_clean["TransactionDate"] = pd.to_datetime(
    transactions_clean["TransactionDate"],
    errors="coerce"
)

transactions_clean["TransactionDate_Status"] = transactions_clean[
    "TransactionDate"
].apply(
    lambda x: "Missing - Review Required" if pd.isna(x) else "Valid"
)

print(
    transactions_clean["TransactionDate_Status"].value_counts()
)

TransactionDate_Status
Valid                        48510
Missing - Review Required      990
Name: count, dtype: int64


In [21]:
# ETL Transformation - Flag Future Transaction Dates

today = pd.Timestamp.today()

transactions_clean["TransactionDate_Status"] = transactions_clean[
    "TransactionDate"
].apply(
    lambda x: "Missing - Review Required"
    if pd.isna(x)
    else "Future Date - Review Required"
    if x > today
    else "Valid"
)

print(transactions_clean["TransactionDate_Status"].value_counts())

TransactionDate_Status
Valid                            48494
Missing - Review Required          990
Future Date - Review Required       16
Name: count, dtype: int64


In [22]:
# ETL Transformation - Loan Date Consistency

loans_clean = data["loans"].copy()

loans_clean["StartDate"] = pd.to_datetime(
    loans_clean["StartDate"],
    errors="coerce"
)

loans_clean["EstimatedEndDate"] = pd.to_datetime(
    loans_clean["EstimatedEndDate"],
    errors="coerce"
)

loans_clean["LoanDate_Status"] = loans_clean.apply(
    lambda row:
        "Missing Date - Review Required"
        if pd.isna(row["StartDate"]) or pd.isna(row["EstimatedEndDate"])
        else "Invalid Date Order - Review Required"
        if row["EstimatedEndDate"] < row["StartDate"]
        else "Valid",
    axis=1
)

print(loans_clean["LoanDate_Status"].value_counts())

LoanDate_Status
Valid                                   320
Missing Date - Review Required           12
Invalid Date Order - Review Required      1
Name: count, dtype: int64


In [23]:
# ETL Transformation - Customer Data Quality Status

customers_clean = data["customers"].copy()

customers_clean["Customer_Data_Status"] = customers_clean.apply(
    lambda row:
        "Both Names Missing - Review Required"
        if pd.isna(row["FirstName"]) and pd.isna(row["LastName"])
        else "FirstName Missing - Review Required"
        if pd.isna(row["FirstName"])
        else "LastName Missing - Review Required"
        if pd.isna(row["LastName"])
        else "Valid",
    axis=1
)

print(customers_clean["Customer_Data_Status"].value_counts())

Customer_Data_Status
Valid                                   1067
LastName Missing - Review Required        22
FirstName Missing - Review Required       21
Both Names Missing - Review Required       1
Name: count, dtype: int64


In [24]:
# ETL Transformation - Address Data Quality Status

addresses_clean = data["addresses"].copy()

addresses_clean["Address_Data_Status"] = addresses_clean.apply(
    lambda row:
        "All Address Fields Missing - Review Required"
        if pd.isna(row["Street"]) and pd.isna(row["City"]) and pd.isna(row["Country"])
        else "Street Missing - Review Required"
        if pd.isna(row["Street"])
        else "City Missing - Review Required"
        if pd.isna(row["City"])
        else "Country Missing - Review Required"
        if pd.isna(row["Country"])
        else "Valid",
    axis=1
)

print(addresses_clean["Address_Data_Status"].value_counts())

Address_Data_Status
Valid                                1149
City Missing - Review Required         26
Street Missing - Review Required       24
Country Missing - Review Required      23
Name: count, dtype: int64


In [25]:
# ETL Transformation - Account Opening Date Status

accounts_clean = data["accounts"].copy()

accounts_clean["OpeningDate"] = pd.to_datetime(
    accounts_clean["OpeningDate"],
    errors="coerce"
)

accounts_clean["OpeningDate_Status"] = accounts_clean[
    "OpeningDate"
].apply(
    lambda x:
        "Missing - Review Required"
        if pd.isna(x)
        else "Valid"
)

print(accounts_clean["OpeningDate_Status"].value_counts())

OpeningDate_Status
Valid                        1634
Missing - Review Required      33
Name: count, dtype: int64


In [26]:
# Final Transaction Cleaning Verification

print("Total Rows:", len(transactions_clean))
print("Duplicate Rows:", transactions_clean.duplicated().sum())
print("Missing Transaction Dates:", transactions_clean["TransactionDate"].isna().sum())
print(
    "Future Transaction Dates:",
    (transactions_clean["TransactionDate"] > pd.Timestamp.today()).sum()
)

Total Rows: 49500
Duplicate Rows: 0
Missing Transaction Dates: 990
Future Transaction Dates: 16


In [27]:
# Create Cleaned Data Folder

cleaned_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Cleaned_Data"

os.makedirs(cleaned_path, exist_ok=True)

print("Cleaned Data folder ready:")
print(cleaned_path)

Cleaned Data folder ready:
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Cleaned_Data


In [28]:
# Save Cleaned DataFrames as CSV

transactions_clean.to_csv(
    os.path.join(cleaned_path, "transactions_clean.csv"),
    index=False
)

loans_clean.to_csv(
    os.path.join(cleaned_path, "loans_clean.csv"),
    index=False
)

customers_clean.to_csv(
    os.path.join(cleaned_path, "customers_clean.csv"),
    index=False
)

addresses_clean.to_csv(
    os.path.join(cleaned_path, "addresses_clean.csv"),
    index=False
)

accounts_clean.to_csv(
    os.path.join(cleaned_path, "accounts_clean.csv"),
    index=False
)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.


In [29]:
print(os.listdir(cleaned_path))

['accounts_clean.csv', 'addresses_clean.csv', 'customers_clean.csv', 'loans_clean.csv', 'transactions_clean.csv']


In [30]:
# Copy Reference Tables Without Changes

reference_tables = [
    "account_statuses",
    "account_types",
    "branches",
    "customer_types",
    "loan_statuses",
    "transaction_types"
]

for table in reference_tables:
    data[table].to_csv(
        os.path.join(cleaned_path, table + "_clean.csv"),
        index=False
    )

print("Reference tables copied successfully.")

Reference tables copied successfully.


In [31]:
print(os.listdir(cleaned_path))

['accounts_clean.csv', 'account_statuses_clean.csv', 'account_types_clean.csv', 'addresses_clean.csv', 'branches_clean.csv', 'customers_clean.csv', 'customer_types_clean.csv', 'loans_clean.csv', 'loan_statuses_clean.csv', 'transactions_clean.csv', 'transaction_types_clean.csv']


In [32]:
# Metadata Extraction for Cleaned Datasets

metadata = []

for table in reference_tables + [
    "accounts",
    "addresses",
    "customers",
    "loans",
    "transactions"
]:
    
    df = data[table]
    
    for column in df.columns:
        metadata.append({
            "Source_Table": table,
            "Source_Column": column,
            "Data_Type": str(df[column].dtype),
            "Nullable": df[column].isnull().any(),
            "Row_Count": len(df)
        })

metadata_df = pd.DataFrame(metadata)

metadata_df.head(20)

,Source_Table,Source_Column,Data_Type,Nullable,Row_Count
0,account_statuses,AccountStatusID,int64,False,3
1,account_statuses,StatusName,object,False,3
2,account_types,AccountTypeID,int64,False,5
3,account_types,TypeName,object,False,5
4,branches,BranchID,int64,False,50
5,branches,BranchName,object,False,50
6,branches,AddressID,int64,False,50
7,customer_types,CustomerTypeID,int64,False,3
8,customer_types,TypeName,object,False,3
9,loan_statuses,LoanStatusID,int64,False,3


In [33]:
# Step 31 - Save Metadata Documentation

documentation_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation"

os.makedirs(documentation_path, exist_ok=True)

metadata_file = os.path.join(
    documentation_path,
    "Data_Metadata.xlsx"
)

metadata_df.to_excel(
    metadata_file,
    index=False
)

print("Metadata documentation saved successfully.")
print(metadata_file)

Metadata documentation saved successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation\Data_Metadata.xlsx


In [34]:
# Step 31A - Create Source-to-Target Mapping

target_mapping = []

target_table_map = {
    "account_statuses": "dim_account_status",
    "account_types": "dim_account_type",
    "branches": "dim_branch",
    "customer_types": "dim_customer_type",
    "loan_statuses": "dim_loan_status",
    "transaction_types": "dim_transaction_type",
    "customers": "dim_customer",
    "addresses": "dim_address",
    "accounts": "dim_account",
    "loans": "dim_loan",
    "transactions": "fact_transaction"
}

for _, row in metadata_df.iterrows():

    source_table = row["Source_Table"]
    source_column = row["Source_Column"]

    target_mapping.append({
        "Source_Table": source_table,
        "Source_Column": source_column,
        "Source_Data_Type": row["Data_Type"],
        "Target_Table": target_table_map[source_table],
        "Target_Column": source_column,
        "Transformation_Rule": "No transformation - direct mapping",
        "Data_Quality_Rule": "Validate based on DQ assessment"
    })

source_target_mapping = pd.DataFrame(target_mapping)

source_target_mapping.head(20)

,Source_Table,Source_Column,Source_Data_Type,Target_Table,Target_Column,Transformation_Rule,Data_Quality_Rule
0,account_statuses,AccountStatusID,int64,dim_account_status,AccountStatusID,No transformation - direct mapping,Validate based on DQ assessment
1,account_statuses,StatusName,object,dim_account_status,StatusName,No transformation - direct mapping,Validate based on DQ assessment
2,account_types,AccountTypeID,int64,dim_account_type,AccountTypeID,No transformation - direct mapping,Validate based on DQ assessment
3,account_types,TypeName,object,dim_account_type,TypeName,No transformation - direct mapping,Validate based on DQ assessment
4,branches,BranchID,int64,dim_branch,BranchID,No transformation - direct mapping,Validate based on DQ assessment
5,branches,BranchName,object,dim_branch,BranchName,No transformation - direct mapping,Validate based on DQ assessment
6,branches,AddressID,int64,dim_branch,AddressID,No transformation - direct mapping,Validate based on DQ assessment
7,customer_types,CustomerTypeID,int64,dim_customer_type,CustomerTypeID,No transformation - direct mapping,Validate based on DQ assessment
8,customer_types,TypeName,object,dim_customer_type,TypeName,No transformation - direct mapping,Validate based on DQ assessment
9,loan_statuses,LoanStatusID,int64,dim_loan_status,LoanStatusID,No transformation - direct mapping,Validate based on DQ assessment


In [35]:
# ============================================================
# DATA LINEAGE DOCUMENTATION
# ============================================================

import pandas as pd
import os

# Existing Source-to-Target Mapping dataframe वापरणे
lineage = source_target_mapping.copy()

# Data Lineage columns तयार करणे
data_lineage = lineage[
    [
        "Source_Table",
        "Source_Column",
        "Target_Table",
        "Target_Column",
        "Transformation_Rule",
        "Data_Quality_Rule"
    ]
].copy()

# Add lineage stages
data_lineage.insert(
    0,
    "Source_Layer",
    "Raw_Data"
)

data_lineage.insert(
    4,
    "Processing_Layer",
    "ETL / Data Quality"
)

data_lineage.insert(
    5,
    "Target_Layer",
    "Target / Analytics"
)

# Display Data Lineage
data_lineage.head(20)

,Source_Layer,Source_Table,Source_Column,Target_Table,Processing_Layer,Target_Layer,Target_Column,Transformation_Rule,Data_Quality_Rule
0,Raw_Data,account_statuses,AccountStatusID,dim_account_status,ETL / Data Quality,Target / Analytics,AccountStatusID,No transformation - direct mapping,Validate based on DQ assessment
1,Raw_Data,account_statuses,StatusName,dim_account_status,ETL / Data Quality,Target / Analytics,StatusName,No transformation - direct mapping,Validate based on DQ assessment
2,Raw_Data,account_types,AccountTypeID,dim_account_type,ETL / Data Quality,Target / Analytics,AccountTypeID,No transformation - direct mapping,Validate based on DQ assessment
3,Raw_Data,account_types,TypeName,dim_account_type,ETL / Data Quality,Target / Analytics,TypeName,No transformation - direct mapping,Validate based on DQ assessment
4,Raw_Data,branches,BranchID,dim_branch,ETL / Data Quality,Target / Analytics,BranchID,No transformation - direct mapping,Validate based on DQ assessment
5,Raw_Data,branches,BranchName,dim_branch,ETL / Data Quality,Target / Analytics,BranchName,No transformation - direct mapping,Validate based on DQ assessment
6,Raw_Data,branches,AddressID,dim_branch,ETL / Data Quality,Target / Analytics,AddressID,No transformation - direct mapping,Validate based on DQ assessment
7,Raw_Data,customer_types,CustomerTypeID,dim_customer_type,ETL / Data Quality,Target / Analytics,CustomerTypeID,No transformation - direct mapping,Validate based on DQ assessment
8,Raw_Data,customer_types,TypeName,dim_customer_type,ETL / Data Quality,Target / Analytics,TypeName,No transformation - direct mapping,Validate based on DQ assessment
9,Raw_Data,loan_statuses,LoanStatusID,dim_loan_status,ETL / Data Quality,Target / Analytics,LoanStatusID,No transformation - direct mapping,Validate based on DQ assessment


In [36]:
# ============================================================
# SAVE DATA LINEAGE DOCUMENTATION
# ============================================================

documentation_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation"

os.makedirs(documentation_path, exist_ok=True)

lineage_file = os.path.join(
    documentation_path,
    "Data_Lineage.xlsx"
)

data_lineage.to_excel(
    lineage_file,
    index=False
)

print("Data Lineage documentation saved successfully.")
print(lineage_file)

Data Lineage documentation saved successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation\Data_Lineage.xlsx


In [37]:
# ============================================================
# TARGET DATA MODEL / STAR SCHEMA DESIGN
# ============================================================

import pandas as pd
import os

# Target table definitions
target_model = [

    {
        "Target_Table": "dim_customer",
        "Table_Type": "Dimension",
        "Source_Table": "customers",
        "Business_Purpose": "Stores customer master information",
        "Primary_Key": "CustomerID"
    },

    {
        "Target_Table": "dim_customer_type",
        "Table_Type": "Dimension",
        "Source_Table": "customer_types",
        "Business_Purpose": "Stores customer type information",
        "Primary_Key": "CustomerTypeID"
    },

    {
        "Target_Table": "dim_account",
        "Table_Type": "Dimension",
        "Source_Table": "accounts",
        "Business_Purpose": "Stores bank account information",
        "Primary_Key": "AccountID"
    },

    {
        "Target_Table": "dim_account_type",
        "Table_Type": "Dimension",
        "Source_Table": "account_types",
        "Business_Purpose": "Stores account type information",
        "Primary_Key": "AccountTypeID"
    },

    {
        "Target_Table": "dim_account_status",
        "Table_Type": "Dimension",
        "Source_Table": "account_statuses",
        "Business_Purpose": "Stores account status information",
        "Primary_Key": "AccountStatusID"
    },

    {
        "Target_Table": "dim_address",
        "Table_Type": "Dimension",
        "Source_Table": "addresses",
        "Business_Purpose": "Stores customer/address information",
        "Primary_Key": "AddressID"
    },

    {
        "Target_Table": "dim_branch",
        "Table_Type": "Dimension",
        "Source_Table": "branches",
        "Business_Purpose": "Stores branch information",
        "Primary_Key": "BranchID"
    },

    {
        "Target_Table": "dim_loan",
        "Table_Type": "Dimension",
        "Source_Table": "loans",
        "Business_Purpose": "Stores loan information",
        "Primary_Key": "LoanID"
    },

    {
        "Target_Table": "dim_loan_status",
        "Table_Type": "Dimension",
        "Source_Table": "loan_statuses",
        "Business_Purpose": "Stores loan status information",
        "Primary_Key": "LoanStatusID"
    },

    {
        "Target_Table": "dim_transaction_type",
        "Table_Type": "Dimension",
        "Source_Table": "transaction_types",
        "Business_Purpose": "Stores transaction type information",
        "Primary_Key": "TransactionTypeID"
    },

    {
        "Target_Table": "fact_transaction",
        "Table_Type": "Fact",
        "Source_Table": "transactions",
        "Business_Purpose": "Stores transactional banking activity for analytics",
        "Primary_Key": "TransactionID"
    }
]

target_model_df = pd.DataFrame(target_model)

target_model_df

,Target_Table,Table_Type,Source_Table,Business_Purpose,Primary_Key
0,dim_customer,Dimension,customers,Stores customer master information,CustomerID
1,dim_customer_type,Dimension,customer_types,Stores customer type information,CustomerTypeID
2,dim_account,Dimension,accounts,Stores bank account information,AccountID
3,dim_account_type,Dimension,account_types,Stores account type information,AccountTypeID
4,dim_account_status,Dimension,account_statuses,Stores account status information,AccountStatusID
5,dim_address,Dimension,addresses,Stores customer/address information,AddressID
6,dim_branch,Dimension,branches,Stores branch information,BranchID
7,dim_loan,Dimension,loans,Stores loan information,LoanID
8,dim_loan_status,Dimension,loan_statuses,Stores loan status information,LoanStatusID
9,dim_transaction_type,Dimension,transaction_types,Stores transaction type information,TransactionTypeID


In [38]:

# ============================================================
# CREATE TARGET DATA FOLDER
# ============================================================

target_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Target_Data"

os.makedirs(target_path, exist_ok=True)

print("Target Data folder ready:")
print(target_path)

Target Data folder ready:
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Target_Data


In [39]:
# ============================================================
# CREATE DIMENSION TABLES
# ============================================================

cleaned_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Cleaned_Data"

# Read cleaned datasets
customers_clean = pd.read_csv(
    os.path.join(cleaned_path, "customers_clean.csv")
)

customer_types_clean = pd.read_csv(
    os.path.join(cleaned_path, "customer_types_clean.csv")
)

accounts_clean = pd.read_csv(
    os.path.join(cleaned_path, "accounts_clean.csv")
)

account_types_clean = pd.read_csv(
    os.path.join(cleaned_path, "account_types_clean.csv")
)

account_statuses_clean = pd.read_csv(
    os.path.join(cleaned_path, "account_statuses_clean.csv")
)

addresses_clean = pd.read_csv(
    os.path.join(cleaned_path, "addresses_clean.csv")
)

branches_clean = pd.read_csv(
    os.path.join(cleaned_path, "branches_clean.csv")
)

loans_clean = pd.read_csv(
    os.path.join(cleaned_path, "loans_clean.csv")
)

loan_statuses_clean = pd.read_csv(
    os.path.join(cleaned_path, "loan_statuses_clean.csv")
)

transaction_types_clean = pd.read_csv(
    os.path.join(cleaned_path, "transaction_types_clean.csv")
)

# ------------------------------------------------------------
# Dimension tables
# ------------------------------------------------------------

dim_customer = customers_clean.drop_duplicates(
    subset=["CustomerID"]
)

dim_customer_type = customer_types_clean.drop_duplicates(
    subset=["CustomerTypeID"]
)

dim_account = accounts_clean.drop_duplicates(
    subset=["AccountID"]
)

dim_account_type = account_types_clean.drop_duplicates(
    subset=["AccountTypeID"]
)

dim_account_status = account_statuses_clean.drop_duplicates(
    subset=["AccountStatusID"]
)

dim_address = addresses_clean.drop_duplicates(
    subset=["AddressID"]
)

dim_branch = branches_clean.drop_duplicates(
    subset=["BranchID"]
)

dim_loan = loans_clean.drop_duplicates(
    subset=["LoanID"]
)

dim_loan_status = loan_statuses_clean.drop_duplicates(
    subset=["LoanStatusID"]
)

dim_transaction_type = transaction_types_clean.drop_duplicates(
    subset=["TransactionTypeID"]
)

# ------------------------------------------------------------
# Save dimension tables
# ------------------------------------------------------------

dimension_tables = {
    "dim_customer": dim_customer,
    "dim_customer_type": dim_customer_type,
    "dim_account": dim_account,
    "dim_account_type": dim_account_type,
    "dim_account_status": dim_account_status,
    "dim_address": dim_address,
    "dim_branch": dim_branch,
    "dim_loan": dim_loan,
    "dim_loan_status": dim_loan_status,
    "dim_transaction_type": dim_transaction_type
}

for table_name, df in dimension_tables.items():

    file_path = os.path.join(
        target_path,
        table_name + ".csv"
    )

    df.to_csv(
        file_path,
        index=False
    )

    print(
        table_name,
        "→",
        len(df),
        "rows saved"
    )

dim_customer → 1100 rows saved
dim_customer_type → 3 rows saved
dim_account → 1651 rows saved
dim_account_type → 5 rows saved
dim_account_status → 3 rows saved
dim_address → 1210 rows saved
dim_branch → 50 rows saved
dim_loan → 330 rows saved
dim_loan_status → 3 rows saved
dim_transaction_type → 4 rows saved


In [40]:
# ============================================================
# FACT TABLE - TRANSACTION
# ============================================================

transactions_clean = data["transactions"].copy()

# Remove exact duplicate transactions
fact_transaction = transactions_clean.drop_duplicates().copy()

# Keep required transaction fields
fact_transaction = fact_transaction[
    [
        "TransactionID",
        "AccountOriginID",
        "AccountDestinationID",
        "TransactionTypeID",
        "Amount",
        "TransactionDate",
        "BranchID",
        "Description"
    ]
]

# Add Transaction Date Status
today = pd.Timestamp.today().normalize()

fact_transaction["TransactionDate"] = pd.to_datetime(
    fact_transaction["TransactionDate"],
    errors="coerce"
)

fact_transaction["TransactionDate_Status"] = fact_transaction[
    "TransactionDate"
].apply(
    lambda x:
        "Missing - Review Required"
        if pd.isna(x)
        else "Future Date - Review Required"
        if x > today
        else "Valid"
)

print("Fact Transaction Rows:", len(fact_transaction))
print()
print(fact_transaction["TransactionDate_Status"].value_counts())

# Display sample
fact_transaction.head()

Fact Transaction Rows: 49500

TransactionDate_Status
Valid                            48493
Missing - Review Required          990
Future Date - Review Required       17
Name: count, dtype: int64


,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description,TransactionDate_Status
0,3022681,201164,200868,2,855.17,2023-04-20 02:00:00,41,Transaction 22681,Valid
1,3037846,200138,201402,2,806.20,2021-08-10 15:00:00,43,Transaction 37846,Valid
2,3045293,201002,201180,1,1229.44,2020-08-16 03:00:00,5,Transaction 45293,Valid
3,3017397,201066,201144,4,4441.60,2021-10-10 06:00:00,14,Transaction 17397,Valid
4,3016750,200289,201413,3,2526.20,2022-07-28 00:00:00,37,Transaction 16750,Valid


In [41]:
# ============================================================
# SAVE FACT TABLE
# ============================================================

fact_file = os.path.join(
    target_path,
    "fact_transaction.csv"
)

fact_transaction.to_csv(
    fact_file,
    index=False
)

print("fact_transaction →", len(fact_transaction), "rows saved")
print(fact_file)

fact_transaction → 49500 rows saved
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Target_Data\fact_transaction.csv


In [42]:
# ============================================================
# FINAL TARGET DATA VALIDATION
# ============================================================

print("=" * 60)
print("FINAL TARGET DATA VALIDATION")
print("=" * 60)

# Load fact table
fact_transaction = pd.read_csv(
    os.path.join(target_path, "fact_transaction.csv")
)

print("\nFact Transaction Validation")
print("-" * 60)

print("Total Rows:", len(fact_transaction))
print("Duplicate Rows:", fact_transaction.duplicated().sum())
print("Unique Transaction IDs:", fact_transaction["TransactionID"].nunique())
print("Missing Transaction IDs:", fact_transaction["TransactionID"].isna().sum())

print("\nTransaction Amount Validation")
print("-" * 60)

print("Negative Amounts:", (fact_transaction["Amount"] < 0).sum())
print("Zero Amounts:", (fact_transaction["Amount"] == 0).sum())

print("\nTransaction Date Validation")
print("-" * 60)

print(
    fact_transaction["TransactionDate_Status"].value_counts()
)

print("\nReferential Integrity Validation")
print("-" * 60)

print(
    "Origin Account Invalid:",
    (~fact_transaction["AccountOriginID"].isin(
        dim_account["AccountID"]
    )).sum()
)

print(
    "Destination Account Invalid:",
    (~fact_transaction["AccountDestinationID"].isin(
        dim_account["AccountID"]
    )).sum()
)

print(
    "Transaction Type Invalid:",
    (~fact_transaction["TransactionTypeID"].isin(
        dim_transaction_type["TransactionTypeID"]
    )).sum()
)

print(
    "Branch Invalid:",
    (~fact_transaction["BranchID"].isin(
        dim_branch["BranchID"]
    )).sum()
)

FINAL TARGET DATA VALIDATION

Fact Transaction Validation
------------------------------------------------------------
Total Rows: 49500
Duplicate Rows: 0
Unique Transaction IDs: 49500
Missing Transaction IDs: 0

Transaction Amount Validation
------------------------------------------------------------
Negative Amounts: 0
Zero Amounts: 0

Transaction Date Validation
------------------------------------------------------------
TransactionDate_Status
Valid                            48493
Missing - Review Required          990
Future Date - Review Required       17
Name: count, dtype: int64

Referential Integrity Validation
------------------------------------------------------------
Origin Account Invalid: 0
Destination Account Invalid: 0
Transaction Type Invalid: 0
Branch Invalid: 0


In [44]:
# Reproducible Transaction Date Validation

# Convert TransactionDate back to datetime
fact_transaction["TransactionDate"] = pd.to_datetime(
    fact_transaction["TransactionDate"],
    errors="coerce"
)

# Fixed validation date
validation_date = pd.Timestamp("2026-08-21")

# Final status classification
fact_transaction["TransactionDate_Status_Final"] = fact_transaction[
    "TransactionDate"
].apply(
    lambda x:
        "Missing - Review Required"
        if pd.isna(x)
        else "Future Date - Review Required"
        if x > validation_date
        else "Valid"
)

print(fact_transaction["TransactionDate_Status_Final"].value_counts())

TransactionDate_Status_Final
Valid                            48493
Missing - Review Required          990
Future Date - Review Required       17
Name: count, dtype: int64


In [45]:
# ============================================================
# FINAL ETL RECONCILIATION
# ============================================================

reconciliation = pd.DataFrame([
    {
        "Dataset": "transactions",
        "Raw_Rows": 50000,
        "Records_Removed": 500,
        "Cleaned_Rows": 49500,
        "Target_Rows": len(fact_transaction),
        "Reconciliation_Status": "PASS"
        if len(fact_transaction) == 49500
        else "FAIL"
    }
])

reconciliation

,Dataset,Raw_Rows,Records_Removed,Cleaned_Rows,Target_Rows,Reconciliation_Status
0,transactions,50000,500,49500,49500,PASS


In [46]:
# ============================================================
# SAVE ETL RECONCILIATION
# ============================================================

reconciliation_file = os.path.join(
    documentation_path,
    "ETL_Reconciliation.xlsx"
)

reconciliation.to_excel(
    reconciliation_file,
    index=False
)

print("ETL Reconciliation saved successfully.")
print(reconciliation_file)

ETL Reconciliation saved successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation\ETL_Reconciliation.xlsx


In [47]:
# ============================================================
# SAVE SOURCE-TO-TARGET MAPPING
# ============================================================

mapping_file = os.path.join(
    documentation_path,
    "Source_to_Target_Mapping.xlsx"
)

source_target_mapping.to_excel(
    mapping_file,
    index=False
)

print("Source-to-Target Mapping saved successfully.")
print(mapping_file)

Source-to-Target Mapping saved successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation\Source_to_Target_Mapping.xlsx


In [48]:
# ============================================================
# SAVE DATA QUALITY FINDINGS
# ============================================================

dq_file = os.path.join(
    documentation_path,
    "Data_Quality_Findings.xlsx"
)

dq_findings.to_excel(
    dq_file,
    index=False
)

print("Data Quality Findings saved successfully.")
print(dq_file)

Data Quality Findings saved successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\Documentation\Data_Quality_Findings.xlsx


In [50]:
# ============================================================
# CREATE PROJECT README
# ============================================================

project_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics"

readme_lines = [
    "# Banking Data Integration & Analytics",
    "",
    "## Project Overview",
    "",
    "This project demonstrates an end-to-end banking data integration and analytics workflow.",
    "",
    "The project focuses on data profiling, data quality assessment, ETL-based data cleaning,",
    "metadata management, source-to-target mapping, data lineage, target data modeling,",
    "and validation of analytics-ready data.",
    "",
    "## Business Problem",
    "",
    "Banking data is distributed across multiple source tables such as customers, accounts,",
    "transactions, branches, loans, and reference tables.",
    "",
    "Before this data can be used for analytics and reporting, it needs to be profiled,",
    "validated, cleaned, transformed, mapped, documented, and validated after ETL processing.",
    "",
    "## Objectives",
    "",
    "- Profile raw banking datasets",
    "- Identify data quality issues",
    "- Perform completeness, uniqueness, validity, consistency, and referential integrity checks",
    "- Document data quality findings",
    "- Apply ETL cleaning and transformation rules",
    "- Create metadata documentation",
    "- Create source-to-target mapping",
    "- Document data lineage",
    "- Design a target dimensional model",
    "- Create dimension and fact tables",
    "- Perform final validation and reconciliation",
    "",
    "## Tools & Technologies",
    "",
    "- Python",
    "- Pandas",
    "- Jupyter Notebook",
    "- SQL",
    "- Microsoft Excel",
    "- Power BI",
    "- CSV",
    "",
    "## Source Datasets",
    "",
    "- Customers",
    "- Customer Types",
    "- Addresses",
    "- Accounts",
    "- Account Types",
    "- Account Statuses",
    "- Branches",
    "- Loans",
    "- Loan Statuses",
    "- Transactions",
    "- Transaction Types",
    "",
    "## ETL Workflow",
    "",
    "```text",
    "Raw Data",
    "    |",
    "    v",
    "Data Profiling",
    "    |",
    "    v",
    "Data Quality Assessment",
    "    |",
    "    v",
    "DQ Findings",
    "    |",
    "    v",
    "ETL Cleaning & Transformation",
    "    |",
    "    v",
    "Cleaned Data",
    "    |",
    "    v",
    "Metadata / Source-to-Target Mapping / Data Lineage",
    "    |",
    "    v",
    "Target Data Model",
    "    |",
    "    v",
    "Dimension + Fact Tables",
    "    |",
    "    v",
    "Final Validation",
    "    |",
    "    v",
    "ETL Reconciliation",
    "```",
    "",
    "## Data Quality Assessment",
    "",
    "The following checks were performed:",
    "",
    "### Completeness",
    "",
    "Missing values were identified in account, customer, address, loan, and transaction data.",
    "",
    "### Uniqueness",
    "",
    "Duplicate identifiers and duplicate transaction records were identified.",
    "",
    "The transaction dataset contained 500 exact duplicate rows.",
    "",
    "### Referential Integrity",
    "",
    "The following relationships were validated:",
    "",
    "- Transaction -> Origin Account",
    "- Transaction -> Destination Account",
    "- Transaction -> Transaction Type",
    "- Transaction -> Branch",
    "- Account -> Customer",
    "",
    "All tested referential integrity checks passed.",
    "",
    "### Validity",
    "",
    "Transaction amounts were checked for negative and zero values.",
    "",
    "Result:",
    "",
    "- Negative Amounts: 0",
    "- Zero Amounts: 0",
    "",
    "### Consistency",
    "",
    "Loan dates were checked to ensure that the estimated end date is not before the start date.",
    "",
    "One invalid loan date-order record was identified and flagged for review.",
    "",
    "## ETL Transformations",
    "",
    "### Transaction Data",
    "",
    "Raw transaction rows: 50,000",
    "",
    "Exact duplicate rows removed: 500",
    "",
    "Cleaned transaction rows: 49,500",
    "",
    "### Missing Transaction Dates",
    "",
    "Missing transaction dates were not artificially populated.",
    "They were flagged as 'Missing - Review Required'.",
    "",
    "### Future Transaction Dates",
    "",
    "Future-dated transactions were retained and flagged for business review.",
    "",
    "### Customer and Address Data",
    "",
    "Missing customer and address attributes were preserved and marked with data-quality status fields.",
    "",
    "### Loan Dates",
    "",
    "Invalid loan date ordering was flagged for business review.",
    "",
    "## Metadata Management",
    "",
    "Metadata was documented using Python and Pandas.",
    "",
    "The metadata documentation contains source table, source column, data type, nullability, and row count information.",
    "",
    "File:",
    "",
    "`Documentation/Data_Metadata.xlsx`",
    "",
    "## Source-to-Target Mapping",
    "",
    "A source-to-target mapping was created to document how source columns map to target tables.",
    "",
    "The mapping includes source table, source column, target table, target column, transformation rules, and data quality rules.",
    "",
    "File:",
    "",
    "`Documentation/Source_to_Target_Mapping.xlsx`",
    "",
    "## Data Lineage",
    "",
    "The project documents the movement of data from source to target:",
    "",
    "```text",
    "Raw CSV Files",
    "      |",
    "      v",
    "Data Quality & ETL Processing",
    "      |",
    "      v",
    "Cleaned Data",
    "      |",
    "      v",
    "Target Dimension / Fact Tables",
    "      |",
    "      v",
    "Analytics",
    "```",
    "",
    "File:",
    "",
    "`Documentation/Data_Lineage.xlsx`",
    "",
    "## Target Data Model",
    "",
    "The target model separates descriptive entities into dimension tables and transactional data into a fact table.",
    "",
    "### Dimension Tables",
    "",
    "- dim_customer",
    "- dim_customer_type",
    "- dim_account",
    "- dim_account_type",
    "- dim_account_status",
    "- dim_address",
    "- dim_branch",
    "- dim_loan",
    "- dim_loan_status",
    "- dim_transaction_type",
    "",
    "### Fact Table",
    "",
    "- fact_transaction",
    "",
    "File:",
    "",
    "`Documentation/Target_Data_Model.xlsx`",
    "",
    "## Target Data Validation",
    "",
    "Final fact transaction table:",
    "",
    "- Total Rows: 49,500",
    "- Duplicate Rows: 0",
    "- Unique Transaction IDs: 49,500",
    "- Missing Transaction IDs: 0",
    "- Negative Amounts: 0",
    "- Zero Amounts: 0",
    "- Invalid Origin Account References: 0",
    "- Invalid Destination Account References: 0",
    "- Invalid Transaction Type References: 0",
    "- Invalid Branch References: 0",
    "",
    "## ETL Reconciliation",
    "",
    "```text",
    "Raw Transactions       : 50,000",
    "Records Removed        : 500",
    "Cleaned Transactions   : 49,500",
    "Target Transactions    : 49,500",
    "Reconciliation Status  : PASS",
    "```",
    "",
    "File:",
    "",
    "`Documentation/ETL_Reconciliation.xlsx`",
    "",
    "## Data Virtualization, Data Mesh & Data-as-a-Product",
    "",
    "These concepts are included as design considerations relevant to modern data integration environments.",
    "",
    "### Data Virtualization",
    "",
    "Data virtualization can provide a logical access layer over distributed banking data without requiring every consumer to physically copy source data.",
    "",
    "### Data Mesh",
    "",
    "The project can be extended toward a data mesh approach by treating Customers, Accounts, Loans, and Transactions as governed business data domains.",
    "",
    "### Data-as-a-Product",
    "",
    "The documented datasets can be treated as reusable data products with defined schemas, metadata, quality rules, and lineage.",
    "",
    "These are documented as concepts and design considerations, not as production implementations.",
    "",
    "## Project Structure",
    "",
    "```text",
    "Banking_Data_Integration_Analytics/",
    "|",
    "+-- Raw_Data/",
    "|",
    "+-- Cleaned_Data/",
    "|",
    "+-- Target_Data/",
    "|",
    "+-- Documentation/",
    "|   +-- Data_Metadata.xlsx",
    "|   +-- Data_Quality_Findings.xlsx",
    "|   +-- Source_to_Target_Mapping.xlsx",
    "|   +-- Data_Lineage.xlsx",
    "|   +-- Target_Data_Model.xlsx",
    "|   +-- ETL_Reconciliation.xlsx",
    "|",
    "+-- Jupyter_Notebook/",
    "|",
    "+-- README.md",
    "```",
    "",
    "## Key Learning Outcomes",
    "",
    "- Data profiling using Python and Pandas",
    "- Data quality assessment",
    "- ETL data cleaning",
    "- Duplicate detection and removal",
    "- Missing-value handling through status flags",
    "- Referential integrity validation",
    "- Metadata management",
    "- Source-to-target mapping",
    "- Data lineage documentation",
    "- Dimensional data modeling",
    "- Fact and dimension table creation",
    "- ETL reconciliation and validation",
    "- Understanding of Data Virtualization and Data Mesh concepts"
]

readme_content = "\n".join(readme_lines)

readme_file = os.path.join(
    project_path,
    "README.md"
)

with open(readme_file, "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README created successfully.")
print(readme_file)

README created successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\README.md


In [2]:
# ============================================================
# FINAL PROJECT STRUCTURE CHECK
# ============================================================

import os

project_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics"

for root, folders, files in os.walk(project_path):

    # Hide Jupyter temporary checkpoint folders
    folders[:] = [
        folder for folder in folders
        if folder != ".ipynb_checkpoints"
    ]

    level = root.replace(project_path, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}    {file}")

Banking_Data_Integration_Analytics/
    README.md
    Cleaned_Data/
        account_statuses_clean.csv
        account_types_clean.csv
        accounts_clean.csv
        addresses_clean.csv
        branches_clean.csv
        customer_types_clean.csv
        customers_clean.csv
        loan_statuses_clean.csv
        loans_clean.csv
        transaction_types_clean.csv
        transactions_clean.csv
    Documentation/
        Data_Lineage.xlsx
        Data_Metadata.xlsx
        Data_Quality_Findings.xlsx
        ETL_Reconciliation.xlsx
        Source_to_Target_Mapping.xlsx
    Documents/
        Banking_Data_Integration_Business_Problem_Statement.docx
        Banking_Data_Integration_Business_Requirements_Document.docx
    ETL/
        banking_ETL.ipynb
    Raw_Data/
        account_statuses.csv
        account_types.csv
        accounts.csv
        addresses.csv
        branches.csv
        customer_types.csv
        customers.csv
        loan_statuses.csv
        loans.csv
        tran

In [3]:
import os

project_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics"
readme_file = os.path.join(project_path, "README.md")

with open(readme_file, "r", encoding="utf-8") as file:
    readme = file.read()

old_structure = """    "+-- Jupyter_Notebook/",
    "|",
    "+-- README.md","""

new_structure = """    "+-- Documents/",
    "|   +-- Business Problem Statement.docx",
    "|   +-- Business Requirements Document.docx",
    "|",
    "+-- ETL/",
    "|   +-- banking_ETL.ipynb",
    "|",
    "+-- README.md","""

# Direct text replacement
readme = readme.replace(
    "+-- Jupyter_Notebook/\n| \n+-- README.md",
    "+-- Documents/\n|   +-- Business Problem Statement.docx\n|   +-- Business Requirements Document.docx\n|\n+-- ETL/\n|   +-- banking_ETL.ipynb\n|\n+-- README.md"
)

with open(readme_file, "w", encoding="utf-8") as file:
    file.write(readme)

print("README structure updated successfully.")
print(readme_file)

README structure updated successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\README.md


In [4]:
import os

project_path = r"C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics"

gitignore_content = """# Jupyter
.ipynb_checkpoints/

# Python
__pycache__/
*.py[cod]

# OS files
.DS_Store
Thumbs.db

# Temporary files
*.tmp
*.log
"""

gitignore_file = os.path.join(project_path, ".gitignore")

with open(gitignore_file, "w", encoding="utf-8") as file:
    file.write(gitignore_content)

print(".gitignore created successfully.")
print(gitignore_file)

.gitignore created successfully.
C:\Users\akank\intership-june2025\Banking_Data_Integration_Analytics\.gitignore
